In [1]:
import gtfs_kit as gk
import pandas as pd
import networkx as nx

feed = gk.read_feed("../data/gtfs", dist_units="km")

In [2]:
def time_to_seconds(t):
    try:
        h, m, s = map(int, t.split(":"))
        return h * 3600 + m * 60 + s
    except (ValueError, AttributeError):
        return None

G = nx.DiGraph()
st = feed.stop_times.sort_values(["trip_id", "stop_sequence"])

for trip_id, group in st.groupby("trip_id"):
    stops_in_trip = group[["stop_id", "arrival_time", "departure_time"]].values
    for i in range(len(stops_in_trip) - 1):
        stop_a, arr_a, dep_a = stops_in_trip[i]
        stop_b, arr_b, dep_b = stops_in_trip[i + 1]
        dep_sec = time_to_seconds(dep_a)
        arr_sec = time_to_seconds(arr_b)
        if dep_sec is not None and arr_sec is not None:
            travel_time = arr_sec - dep_sec
            if travel_time > 0:
                G.add_edge(stop_a, stop_b, weight=travel_time, trip_id=trip_id)

print("Nodes:", G.number_of_nodes())
print("Edges:", G.number_of_edges())

Nodes: 9875
Edges: 17543


In [3]:
stop_names = feed.stops.set_index("stop_id")["stop_name"].to_dict()

path = nx.shortest_path(G, source="24350", target="25171", weight="weight")
total_time = nx.shortest_path_length(G, source="24350", target="25171", weight="weight")

print([stop_names.get(s, s) for s in path])
print("Total time:", total_time/60, "minutes")

['NICE Bridge', 'Lakshmi Layout', 'Beguru', 'Beguru Lake', 'Aralimara (PK Kalyana Mantapa )', 'Vishwapriya Layout Cross', 'Beguru Canara Bank', 'St Francis School', 'Galaxy Paradise (Tent)', 'Hongasandra', 'Kalyanamantapa (Oxford College)', 'Deccan', 'Bommanahalli', 'Roopena Agrahara', 'Madivala', 'St Johns Hospital', 'Lakkasandra', 'Shanthinagara Bus Station - Platform 2', 'Pallavi Talkies', 'Corporation', 'Cauvery Bhavana', 'Maharani College', 'Telephone Exchange AGO/VSD', 'RM Guttahalli', 'Palace Ground', 'Mekhri Circle', 'Hebbala', 'Raitarasanthe Yelahanka', 'Kogilu Cross', 'Venkatala', 'Palanahalli Gate', 'Bagalur Cross', 'Dwaraka Nagara', 'Kattigenahalli Cross', 'Reva College Gate', 'Country Club Bagaluru', 'Reva College Gate', 'Dwaraka Nagara']
Total time: 116.68333333333334 minutes


In [4]:
import folium

stop_coords = feed.stops.set_index("stop_id")[["stop_lat", "stop_lon"]].to_dict("index")

route_coords = []
for stop_id in path:
    coord = stop_coords.get(stop_id)
    if coord:
        route_coords.append((coord["stop_lat"], coord["stop_lon"]))

mid = route_coords[len(route_coords)//2]
m = folium.Map(location=mid, zoom_start=12)

folium.PolyLine(route_coords, color="blue", weight=4, opacity=0.8).add_to(m)
folium.Marker(route_coords[0], popup=f"Start: {stop_names.get(path[0], path[0])}", icon=folium.Icon(color="green")).add_to(m)
folium.Marker(route_coords[-1], popup=f"End: {stop_names.get(path[-1], path[-1])}", icon=folium.Icon(color="red")).add_to(m)

for stop_id, coord in zip(path[1:-1], route_coords[1:-1]):
    folium.CircleMarker(coord, radius=3, color="orange", fill=True, popup=stop_names.get(stop_id, stop_id)).add_to(m)

m.save("../data/route_map.html")
m